# Textbook quantum Fourier transform

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2023 Google LLC, Apache-2.0. [Locked source notebook](https://github.com/quantumlib/Qualtran/blob/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/bloqs/qft/qft_text_book.ipynb).

In [ ]:
from IPython.display import SVG, display
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from examples.algorithms.field_addition_catalog import field_signature_svg

## Textbook construction

The $n$-qubit transform follows Nielsen and Chuang, chapter 5.1. Each control applies a phase-gradient layer to earlier wires in reverse order, then receives a Hadamard. The optional final swaps reverse wire order. The emitted circuit uses $n$ Hadamards, $n(n-1)/2$ controlled phases, and at most $\lfloor n/2\rfloor$ swaps. It measures no qubits and needs no extra work wires.

The approximate QFT truncates small controlled phases. The phase-gradient QFT uses an additional gradient register and controlled arithmetic. The two-wire fermionic Fourier transform implements a different transform on occupation states.

In [ ]:
from examples.algorithms.textbook_qft_catalog import textbook_qft_examples, textbook_qft_graph
from flagquantum.algorithms.primitives.qft_resource_model import (
    source_textbook_qft_signature, source_textbook_qft_cost, source_textbook_qft_symbolic_formula,
)
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources

### Example instances

The source factory uses three qubits. The symbolic example uses an unrestricted symbol $n$ and supplies metadata without a concrete circuit.

In [ ]:
concrete, symbolic = textbook_qft_examples()
source_textbook_qft_signature(concrete)

In [ ]:
source_textbook_qft_signature(symbolic)

#### Graphical signature

The unsigned register $q$ is a through port.

In [ ]:
display(SVG(field_signature_svg(tuple(
    (f"Textbook Fourier transform ({width})", source_textbook_qft_signature(width), {})
    for width in (concrete, symbolic)
))))

### Call graph

Concrete declarations call each smaller controlled phase gradient once. Symbolic declarations group them under the largest gradient size, with multiplicity $\lfloor n/2\rfloor$. These are source model counts.

In [ ]:
for width in (concrete, symbolic):
    graph = textbook_qft_graph(width)
    display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
    print(graph.leaf_counts)
print("Concrete source model:", estimate_component_resources(source_textbook_qft_cost(concrete)))
print("Symbolic source model:", source_textbook_qft_symbolic_formula(symbolic))

### Execute the transform and its adjoint

The direct circuit uses no ancillas or measurements. Model measurements above belong to a controlled phase-gradient decomposition used for estimation.

In [ ]:
from collections import Counter
import torch
from flagquantum.circuit import Circuit
from flagquantum.algorithms.primitives.qft import qft
from examples.algorithms.binary_field_arithmetic import FieldOperation

forward = qft(concrete)
print("Emitted primitives:", Counter(op.name for op in forward.to_ir().instructions))
values = torch.arange(1, 2**concrete + 1, dtype=torch.float64)
state = torch.complex(values, 2-values)
state /= state.norm()
circuit = Circuit(concrete, inputs=state, dtype=torch.complex128)
for transform in (forward, forward.adjoint()):
    circuit.append(FieldOperation(concrete, tuple(transform.to_ir().instructions)))
torch.testing.assert_close(circuit.state().flatten(), state)